In [1]:
!nvidia-smi

Thu Oct 22 05:00:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.8/79.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.0/31.0 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 93.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 93.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 99.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 93.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 23.8 MB/s eta 0:00:00


In [3]:
import unsloth
print("Unsloth OK")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth OK


In [4]:
!pip install -q pypdf python-docx pandas datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 407.8/407.8 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 27.1 MB/s eta 0:00:00


In [7]:
import os
from pypdf import PdfReader
from docx import Document
import pandas as pd
from datasets import Dataset

def read_file(path):
    ext = path.split(".")[-1].lower()
    if ext == "pdf":
        reader = PdfReader(path)
        return [p.extract_text() or "" for p in reader.pages]
    if ext == "docx":
        doc = Document(path)
        return ["\n".join(p.text for p in doc.paragraphs if p.text)]
    if ext == "txt":
        return [open(path, encoding="utf-8", errors="ignore").read()]
    if ext == "csv":
        df = pd.read_csv(path).fillna("")
        return [" | ".join(f"{c}: {r[c]}" for c in df.columns) for _, r in df.iterrows()]
    return []

def chunk(text, size=1000):
    text = " ".join(text.split())
    return [text[i:i+size] for i in range(0, len(text), size) if len(text[i:i+size]) > 100]

rows = []
for fname in os.listdir("docs"):
    for part in read_file(f"docs/{fname}"):
        for c in chunk(part):
            rows.append({"text": c, "source": fname})

dataset = Dataset.from_list(rows)
print("Files:", os.listdir("docs"))
print("Training chunks:", len(dataset))
print("\nSample:\n", dataset[0]["text"][:300])

Files: ['.ipynb_checkpoints', 'details.pdf']
Training chunks: 751

Sample:
 Cracking the AI System Design Interview An Insider's Guide to AI System Design Interviews, with 10 Real-World Use Cases Written by Aishwarya Naresh Reganti Kiriti Badam Published by LevelUp Labs 2026 • levelup-labs.ai


In [8]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    max_seq_length = 1024,
    dtype = None,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

model.print_trainable_parameters()

==((====))==  Unsloth 2026.10.7: Fast Llama patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-1B-Instruct-bnb-4bit as a legacy tokenizer.
Unsloth 2026.10.7 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


trainable params: 11,272,192 || all params: 1,247,086,592 || trainable%: 0.9039


In [9]:
import torch
from trl import SFTTrainer, SFTConfig

EOS = tokenizer.eos_token
dataset = dataset.map(lambda x: {"text": x["text"] + EOS})

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = True,
        bf16 = False,
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)
print("Trainer ready")

Map:   0%|          | 0/751 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/751 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Trainer ready


In [10]:
trainer_stats = trainer.train()
print("Done in", round(trainer_stats.metrics["train_runtime"] / 60, 1), "minutes")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 751 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Step,Training Loss
5,5.249152
10,4.970391
15,4.642213
20,4.347905
25,4.379855
30,4.209744
35,4.176011
40,4.115561
45,4.096544
50,3.933475


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


Done in 1.4 minutes


In [11]:
trainer.args.max_steps = 200
trainer_stats = trainer.train()
print("Done in", round(trainer_stats.metrics["train_runtime"] / 60, 1), "minutes")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 751 | Num Epochs = 3 | Total steps = 200
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Step,Training Loss
5,3.906352
10,3.858531
15,3.560296
20,3.422544
25,3.575585
30,3.393515
35,3.509261
40,3.374593
45,3.447250
50,3.234349


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-200/tokenizer_config.json.


Done in 3.3 minutes


In [12]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

prompt = "In an AI system design interview, the first step is to"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=120, use_cache=True)
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])

Both `max_new_tokens` (=120) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


In an AI system design interview, the first step is to state the problem and then name the 5 layers above the model. The 5 layers are the structure of the design, and the 4 layers that matter most are the model, the optimizer, the gates, and the evals. Each layer is a decision you make about the problem, and the wrong one will ship in the walls you cannot engineer out. The 5 layers are the structure of the design, and the 4 that matter most are the model, the optimizer, the gates, and the evals. The 5 layers are the structure of the design, and the 


In [13]:
model.save_pretrained("my_lora_adapter")
tokenizer.save_pretrained("my_lora_adapter")

import os
print(os.listdir("my_lora_adapter"))

Unsloth: Restored added_tokens_decoder metadata in my_lora_adapter/tokenizer_config.json.


['README.md', 'tokenizer_config.json', 'chat_template.jinja', 'adapter_config.json', 'adapter_model.safetensors', 'tokenizer.json']


In [14]:
from google.colab import drive
drive.mount("/content/drive")

!cp -r my_lora_adapter /content/drive/MyDrive/my_lora_adapter
!ls /content/drive/MyDrive/my_lora_adapter

Mounted at /content/drive
adapter_config.json	   chat_template.jinja	tokenizer_config.json
adapter_model.safetensors  README.md		tokenizer.json


In [15]:
prompts = [
    "In an AI system design interview, the first step is to",
    "The most important thing to evaluate in an AI system is",
]

for p in prompts:
    inputs = tokenizer(p, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        use_cache=True,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.2,
    )
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))
    print("-" * 60)

Both `max_new_tokens` (=100) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In an AI system design interview, the first step is to read and understand the problem before you start writing code. The 5-layer spine above the model is a framework for doing this, so let’s get it down before we build anything . Problem First (layer 1). Say with me: “Whatistheproblem.” Thework Here is where it shows up in the walls around every case: each one has a single number or quantity that must be reached toward the center , which is what makes them all worth reading together rather than as separate chapters.
------------------------------------------------------------
The most important thing to evaluate in an AI system is the false-approve rate, because it is where a decision that looks good on paper can fail. The 5-layer spine stays intact: data and guardrails, model execution with evals, latency and cost, human review as safety tools, and observability for every node. Where these systems diverge from one another is in their architecture, so pay attention to how each layer m

In [16]:
prompt = "The most important thing to evaluate in an AI system is"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
gen = dict(max_new_tokens=80, do_sample=True, temperature=0.7, top_p=0.9, repetition_penalty=1.2)

with model.disable_adapter():
    base = model.generate(**inputs, **gen)
print("BASE MODEL:\n", tokenizer.decode(base[0], skip_special_tokens=True))

tuned = model.generate(**inputs, **gen)
print("\nFINE-TUNED:\n", tokenizer.decode(tuned[0], skip_special_tokens=True))

Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


BASE MODEL:
 The most important thing to evaluate in an AI system is the **consistency** of its outputs. The goal is not just to make predictions, but also to ensure that they are consistent and fair.

For example:

*   Imagine a chatbot designed for customer service. Its primary function should be to provide accurate information about products or services.
*   A key aspect of this functionality could involve checking if certain questions have been asked before by customers with similar characteristics

FINE-TUNED:
 The most important thing to evaluate in an AI system is the latency and cost per request, because that is how users experience it. The metrics you watch are the number of requests a customer has to wait for your answer before asking another one , which is what turns “answer” into an expensive round trip rather than a single reply. • Reliability:thepercentageofquestionswhoseanswersarecorrectandtimely,thepercentagediscoveryfailures,or
